In [1]:
%%capture
!pip install supervision
!pip install ultralytics
!pip install roboflow
!pip install dagshub
!pip install mlflow

In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="9VCfGRMmzvi61B4c9qu5")
project = rf.workspace("agabaembedded").project("ship-detection-sar-full")
version = project.version(11)
dataset = version.download("yolo26")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Ship-Detection-SAR-Full-11 in yolo26:: 100%|██████████| 23071/23071 [00:05<00:00, 4232.98it/s] 


In [ ]:
import supervision as sv
from ultralytics import YOLO
import os
import cv2

# 1. Setup paths
model = YOLO('https://github.com/AgabaEmbedded/Ship-Detection-in-SAR-Images/releases/download/v2.1/best92.2.pt')
VALID_IMAGES_PATH = "/content/Ship-Detection-SAR-Full-6/valid/images"
OUTPUT_PATH = "inspection_results"

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 2. Initialize Supervision annotators
# Note: In newer versions of supervision, BoxAnnotator and LabelAnnotator
# are often used together for a clean look.
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

# 3. Process images
for image_name in os.listdir(VALID_IMAGES_PATH):
    # Filter to ensure we only process images
    if not image_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp')):
        continue

    img_path = os.path.join(VALID_IMAGES_PATH, image_name)

    # Run Inference
    results = model(img_path)[0]
    detections = sv.Detections.from_ultralytics(results)

    # Create labels for the boxes (Class + Confidence)
    labels = [
        f"{model.model.names[class_id]} {confidence:.2f}"
        for class_id, confidence
        in zip(detections.class_id, detections.confidence)
    ]

    # Annotate the image
    annotated_image = box_annotator.annotate(
        scene=results.orig_img.copy(),
        detections=detections
    )

    annotated_image = label_annotator.annotate(
        scene=annotated_image,
        detections=detections,
        labels=labels
    )

    # 4. Save the image with the same name
    save_path = os.path.join(OUTPUT_PATH, image_name)
    cv2.imwrite(save_path, annotated_image)

print(f"Done! Inspected images saved to: {OUTPUT_PATH}")

In [ ]:
import os
import zipfile

def zip_folder(folder_path, output_zip):
    """
    Compresses the contents of a folder into a ZIP file.

    :param folder_path: Path to the folder to be zipped
    :param output_zip: Path to the output ZIP file
    """
    try:
        # Validate folder path
        if not os.path.exists(folder_path):
            raise FileNotFoundError(f"Folder '{folder_path}' does not exist.")
        if not os.path.isdir(folder_path):
            raise NotADirectoryError(f"'{folder_path}' is not a directory.")

        # Create ZIP file
        with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(folder_path):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Store relative path inside the ZIP
                    arcname = os.path.relpath(file_path, start=folder_path)
                    zipf.write(file_path, arcname)

        print(f"Folder '{folder_path}' successfully zipped to '{output_zip}'.")

    except Exception as e:
        print(f"Error: {e}")

# Example usage
if __name__ == "__main__":
    folder_to_zip = "/content/Ship-Detection-SAR-Full-6/valid/labels"       # Replace with your folder name
    output_zip_file = "labels.zip" # Output ZIP file name
    zip_folder(folder_to_zip, output_zip_file)


Folder '/content/Ship-Detection-SAR-Full-6/valid/labels' successfully zipped to 'labels.zip'.


In [ ]:
import dagshub
import mlflow
from ultralytics import YOLO
from google.colab import userdata
token = userdata.get('dagshub')

# Initialize DagsHub connection
# Replace 'your-username' and 'your-repo-name' with your actual DagsHub details
dagshub.auth.add_app_token(token)
dagshub.init(repo_owner='AgabaEmbedded', repo_name='Ship-Detection-in-SAR-Images')

# Optional: Set the experiment name
mlflow.set_experiment("Ship-Detection-in-SAR-Images")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=cd39c133-1be7-4a70-9e81-c112121fe5c8&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e832459dbf1b642244a3fc6a386256d92bc3029c3d3947585a95a1c6a15f23fd




Accessing as AgabaEmbedded

Initialized MLflow to track repo "AgabaEmbedded/Ship-Detection-in-SAR-Images"

Repository AgabaEmbedded/Ship-Detection-in-SAR-Images initialized!

2026/04/13 13:28:31 INFO mlflow.tracking.fluent: Experiment with name 'Ship-Detection-in-SAR-Images' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/b4f57012b74d42188145fbe55f62659b', creation_time=1776086911857, experiment_id='0', last_update_time=1776086911857, lifecycle_stage='active', name='Ship-Detection-in-SAR-Images', tags={}, trace_location=None, workspace='default'>

In [ ]:
from ultralytics import settings

# Ensure MLflow tracking is enabled
settings.update({'mlflow': True})

In [ ]:
!export DAGSHUB_USER_TOKEN=token

In [ ]:
import mlflow
import dagshub
from ultralytics import YOLO
from google.colab import userdata
token = userdata.get('dagshub')
dagshub.auth.add_app_token(token)
# Initialize DagsHub
dagshub.init(repo_owner='AgabaEmbedded', repo_name='Ship-Detection-val')
#mlflow.set_experiment("SAR-Ship-Detection-YOLOv8")
# Load your best weights
model = YOLO('https://github.com/AgabaEmbedded/Ship-Detection-in-SAR-Images/releases/download/v2.0/best-12.pt')

params={
"conf": 0.0,
"iou": 0.1
         }



# Explicitly wrap in an MLflow run
with mlflow.start_run(run_name="val-new1"):
    # Run validation
    results = model.val(
                        data='/content/Ship-Detection-SAR-Full-1/data.yaml', # Path to your dataset config
                        split='val',              # Use the validation split
                        imgsz=620,                # Ensure this matches your training size
                        save_json=True,
                        **params                 # Additional parameters
                        )

    # Extract and log specific metrics manually
    metrics_to_log = {
        "mAP50": results.box.map50,
        "mAP50-95": results.box.map,
        "precision": results.box.mp,
        "recall": results.box.mr
    }


    mlflow.log_params(params)
    mlflow.log_metrics(metrics_to_log)

    # Log the summary plots (Confusion Matrix, PR Curve) if they exist
    # YOLO saves these to 'runs/detect/val' by default
    mlflow.log_artifacts(results.save_dir)

In [8]:
!yolo val \
model=https://github.com/AgabaEmbedded/Ship-Detection-in-SAR-Images/releases/download/v2.1/best92.2.pt \
data=/content/Ship-Detection-SAR-Full-11/data.yaml \
split=test \
imgsz=640 \
batch=16

Found https://github.com/AgabaEmbedded/Ship-Detection-in-SAR-Images/releases/download/v2.1/best92.2.pt locally at weights/best92.2.pt
Ultralytics 8.4.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 986.7±384.7 MB/s, size: 30.7 KB)
val: Scanning /content/Ship-Detection-SAR-Full-11/test/labels... 461 images, 103 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 461/461 2.1Kit/s 0.2s
val: New cache created: /content/Ship-Detection-SAR-Full-11/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 1.9it/s 15.5s
                   all        461        641      0.931      0.909      0.922      0.425
Speed: 2.0ms preprocess, 27.9ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val-4
💡 Learn more at https://docs.ultralytics.com/mod